# Universal-3.6 Pro Streaming — Speech-to-Text on Amazon SageMaker

This notebook demonstrates end-to-end usage of the **Universal-3.6 Pro Streaming** speech recognition model package from AWS Marketplace: subscribing, deploying a real-time endpoint, live streaming audio, transcribing whole files, running batch transform, reading your usage metrics, and cleaning up.

One endpoint serves both kinds of request:

- **Live streaming** over `InvokeEndpointWithBidirectionalStream`: send audio as it is captured and receive partial and final transcripts while the speaker talks.
- **Whole-file transcription** over `InvokeEndpoint`, asynchronous inference and batch transform.

## Contents

1. [Prerequisites](#prereqs)
2. [Subscribe to the model package](#subscribe)
3. [Setup](#setup)
4. [Deploy a real-time endpoint](#deploy)
5. [Live streaming](#streaming)
6. [Streaming options](#options)
7. [Whole-file transcription](#invoke)
8. [Batch transform](#batch)
9. [Usage metrics](#metrics)
10. [Clean up](#cleanup)
11. [Unsubscribe (optional)](#unsubscribe)

The [README](README.md) is the full reference for the protocol, options, limits and billing.

<a id='prereqs'></a>
## 1. Prerequisites

- An active subscription to **Universal-3.6 Pro Streaming** in AWS Marketplace.
- An IAM role SageMaker can assume, with `AmazonSageMakerFullAccess` (or equivalent permissions to create models, endpoints and transform jobs).
- Service quota for one endpoint instance of a supported type (for example `ml.g6e.xlarge for endpoint usage`), and one `ml.g5.2xlarge` for batch transform; new accounts often start at 0.
- A **Python 3.12+** kernel: live streaming uses AWS's `aws-sdk-sagemaker-runtime-http2` package, which requires it.
- Run this notebook in the same AWS Region as your subscription.

<a id='subscribe'></a>
## 2. Subscribe to the model package

1. Open the **Universal-3.6 Pro Streaming** listing in AWS Marketplace.
2. Read the product overview, pricing and usage information, and review the End User License Agreement.
3. Choose **Continue to Subscribe**, then accept the offer.
4. After subscribing, open **AWS Marketplace > Manage subscriptions** (or the SageMaker console's **Marketplace model packages** page), select this product, and copy the **model package ARN** for your AWS Region.
5. Paste that ARN into `MODEL_PACKAGE_ARN` below.

<a id='setup'></a>
## 3. Setup

In [ ]:
%pip install --quiet -r requirements.txt

In [ ]:
import asyncio
import base64
import contextlib
import json
import time
import urllib.parse
import wave
from datetime import datetime, timedelta, timezone

import boto3
from botocore.config import Config

session = boto3.Session()
region = session.region_name
sm = session.client("sagemaker")
# Wait out InvokeEndpoint's 60 s limit, and don't retry: a retried request is
# transcribed (and billed) again.
smr = session.client("sagemaker-runtime", config=Config(read_timeout=70, retries={"max_attempts": 0}))

# From the AWS Marketplace listing: use the ARN for your Region.
MODEL_PACKAGE_ARN = "<YOUR_MODEL_PACKAGE_ARN>"

# The IAM role SageMaker assumes to run the model. Leave empty in SageMaker
# Studio or a notebook instance to use the notebook's own role.
ROLE_ARN = ""

if not ROLE_ARN:
    # The notebook runs as an assumed role; look up that role's full ARN.
    caller = session.client("sts").get_caller_identity()["Arn"]
    if ":assumed-role/" not in caller:
        raise ValueError("Set ROLE_ARN to a SageMaker execution role")
    role_name = caller.split(":assumed-role/")[1].split("/")[0]
    ROLE_ARN = session.client("iam").get_role(RoleName=role_name)["Role"]["Arn"]

print(f"Region: {region}")

<a id='deploy'></a>
## 4. Deploy a real-time endpoint

> **Pin the inference AMI.** Set `InferenceAmiVersion="al2023-ami-sagemaker-inference-gpu-4-1"` on the production variant. It provides NVIDIA driver 580 (CUDA 13), which the model runs on natively; on an older default driver the container falls back to its CUDA compatibility libraries.

Supported real-time instance types: `ml.g6e.xlarge`, `ml.g6e.2xlarge`, `ml.g6e.4xlarge` (NVIDIA L40S) and `ml.g7e.2xlarge`, `ml.g7e.4xlarge` (NVIDIA RTX PRO 6000). Each has one GPU and serves up to 32 concurrent streams; scale with instance count rather than size.

In [ ]:
MODEL_NAME = "universal-3-6-pro-streaming"
ENDPOINT_CONFIG_NAME = "universal-3-6-pro-streaming-config"
ENDPOINT_NAME = "universal-3-6-pro-streaming-endpoint"

INSTANCE_TYPE = "ml.g6e.xlarge"
INFERENCE_AMI_VERSION = "al2023-ami-sagemaker-inference-gpu-4-1"

In [ ]:
sm.create_model(
    ModelName=MODEL_NAME,
    PrimaryContainer={"ModelPackageName": MODEL_PACKAGE_ARN},
    ExecutionRoleArn=ROLE_ARN,
    EnableNetworkIsolation=True,  # required for AWS Marketplace model packages
)
print(f"Created model: {MODEL_NAME}")

In [ ]:
sm.create_endpoint_config(
    EndpointConfigName=ENDPOINT_CONFIG_NAME,
    ProductionVariants=[
        {
            "VariantName": "primary",
            "ModelName": MODEL_NAME,
            "InitialInstanceCount": 1,
            "InstanceType": INSTANCE_TYPE,
            "InferenceAmiVersion": INFERENCE_AMI_VERSION,
            # The weights are copied to the instance and loaded onto the GPU
            # at startup; allow ample time for both.
            "ModelDataDownloadTimeoutInSeconds": 1800,
            "ContainerStartupHealthCheckTimeoutInSeconds": 1800,
        }
    ],
)
sm.create_endpoint(EndpointName=ENDPOINT_NAME, EndpointConfigName=ENDPOINT_CONFIG_NAME)
print(f"Creating endpoint: {ENDPOINT_NAME}")

Endpoint creation takes several minutes: the instance starts, the image and weights are copied to it, and the model loads onto the GPU and warms up before the endpoint reports `InService`. To let SageMaker pick from several sizes, give the production variant an `InstancePools` list instead of a single `InstanceType`.

In [ ]:
sm.get_waiter("endpoint_in_service").wait(
    EndpointName=ENDPOINT_NAME, WaiterConfig={"Delay": 30, "MaxAttempts": 80}
)
print("Endpoint status:", sm.describe_endpoint(EndpointName=ENDPOINT_NAME)["EndpointStatus"])

<a id='streaming'></a>
## 5. Live streaming

A stream is opened with `InvokeEndpointWithBidirectionalStream` on the SageMaker runtime's HTTP/2 endpoint (port 8443). Session options go in `ModelQueryString` as URL query parameters, and every message is AssemblyAI's [v3 streaming protocol](https://www.assemblyai.com/docs/api-reference/streaming-api/streaming-api):

- **You send** audio as binary payloads (16-bit PCM by default, 20–1000 ms per message), and JSON control messages such as `{"type": "Terminate"}` as UTF-8 payloads.
- **You receive** `Begin` when the session opens, `Turn` messages as the speaker talks (`end_of_turn: false` for partials that will change, `end_of_turn: true` for the final text of a turn), and `Termination` when the session ends.

The helper below streams a WAV file at real time, as a live audio source would, and calls `on_message` for each server message. It uses one SDK client per stream: streams that share a client (one HTTP/2 connection) can stall.

In [ ]:
from aws_sdk_sagemaker_runtime_http2.client import AsyncSageMakerRuntimeHTTP2Client
from aws_sdk_sagemaker_runtime_http2.config import AsyncSageMakerRuntimeHTTP2Config
from aws_sdk_sagemaker_runtime_http2.models import (
    InvokeEndpointWithBidirectionalStreamInput,
    RequestPayloadPart,
    RequestStreamEventPayloadPart,
)
from smithy_http.aio.crt import AWSCRTHTTPClient

FRAME_MS = 50


def read_wav(path):
    """Return (16-bit mono PCM bytes, sample rate) from a WAV file."""
    with wave.open(path) as w:
        assert w.getsampwidth() == 2 and w.getnchannels() == 1, "need 16-bit mono PCM"
        return w.readframes(w.getnframes()), w.getframerate()


def payload(data, data_type):
    return RequestStreamEventPayloadPart(value=RequestPayloadPart(bytes_=data, data_type=data_type))


def quiet_sdk_close(loop, context):
    # After the server ends a stream the SDK still tries to close the request
    # body in the background, which fails harmlessly.
    if "HTTP_STREAM_HAS_COMPLETED" not in str(context.get("exception", "")):
        loop.default_exception_handler(context)


async def stream_audio(endpoint, pcm, sample_rate, params=None, on_message=print):
    """Stream PCM audio in real time. Returns True if the session ended with Termination."""
    creds = session.get_credentials().get_frozen_credentials()
    config = await AsyncSageMakerRuntimeHTTP2Config.resolve(
        region=region,
        endpoint_uri=f"https://runtime.sagemaker.{region}.amazonaws.com:8443",
        transport=AWSCRTHTTPClient(),
        aws_access_key_id=creds.access_key,
        aws_secret_access_key=creds.secret_key,
        aws_session_token=creds.token,
    )
    query = urllib.parse.urlencode(
        {"sample_rate": sample_rate, "encoding": "pcm_s16le", **(params or {})},
        # SageMaker rejects "+" for a space in ModelQueryString; use %20.
        quote_via=urllib.parse.quote,
    )
    asyncio.get_running_loop().set_exception_handler(quiet_sdk_close)

    async with AsyncSageMakerRuntimeHTTP2Client(config=config) as client:
        stream = await client.invoke_endpoint_with_bidirectional_stream(
            InvokeEndpointWithBidirectionalStreamInput(endpoint_name=endpoint, model_query_string=query)
        )

        async def send():
            frame = sample_rate * 2 * FRAME_MS // 1000
            start = time.monotonic()
            for i, offset in enumerate(range(0, len(pcm), frame)):
                await stream.input_stream.send(payload(pcm[offset:offset + frame], "BINARY"))
                await asyncio.sleep(max(0, start + (i + 1) * FRAME_MS / 1000 - time.monotonic()))
            await stream.input_stream.send(payload(b'{"type": "Terminate"}', "UTF8"))

        sender = asyncio.create_task(send())
        terminated = False
        try:
            _, output = await stream.await_output()
            while True:
                try:
                    event = await asyncio.wait_for(output.receive(), 10 if sender.done() else None)
                except Exception as e:
                    # SageMaker reports the container closing the stream (even
                    # normally) as a ModelStreamError; after an Error message
                    # this is how a refused session ends.
                    print(f"Stream ended: {type(e).__name__}")
                    break
                value = getattr(event, "value", None)
                if not hasattr(value, "bytes_"):
                    break
                message = json.loads(value.bytes_)
                on_message(message)
                if message["type"] == "Termination":
                    terminated = True
                    break
        finally:
            sender.cancel()
            await asyncio.gather(sender, return_exceptions=True)
            with contextlib.suppress(Exception):
                await asyncio.wait_for(stream.close(), 2)
    return terminated


def show(message):
    """Print each final turn, and the session's start and end."""
    if message["type"] == "Turn" and message["end_of_turn"]:
        print(f"[{message['turn_order']}] {message['transcript']}")
    elif message["type"] == "Begin":
        print("Session", message["id"])
    elif message["type"] == "Termination":
        print(f"Ended after {message['audio_duration_seconds']} s of audio")
    elif message["type"] == "Error":
        print(f"Error {message.get('error_code')}: {message.get('error')}")

This notebook uses the sample recordings in [`data/input/`](data/input): public-domain LibriVox audiobook clips at 16 kHz mono 16-bit PCM. Stream the 30-second sample; final turns print as the speaker finishes each sentence.

In [ ]:
pcm, sample_rate = read_wav("data/input/sample-30s.wav")
await stream_audio(ENDPOINT_NAME, pcm, sample_rate, on_message=show)

Partial turns arrive while a sentence is still being spoken; each repeats the turn so far, so a live UI replaces the line in place until the final arrives:

In [ ]:
messages = []
await stream_audio(ENDPOINT_NAME, *read_wav("data/input/sample-10s.wav"), on_message=messages.append)

for m in messages:
    if m["type"] == "Turn":
        print(f"[{m['turn_order']}] {'FINAL  ' if m['end_of_turn'] else 'partial'} {m['transcript']}")

final = next(m for m in messages if m["type"] == "Turn" and m["end_of_turn"])
print()
print(json.dumps({k: final[k] for k in ("turn_order", "end_of_turn", "transcript")}, indent=2))
print("first word:", final["words"][0])

<a id='options'></a>
## 6. Streaming options

Pass options as query parameters. `keyterms_prompt` takes a JSON list of words and phrases to recognise, and `mode` trades latency against accuracy (`balanced`, the default, `min_latency` or `max_accuracy`). The README lists every option.

In [ ]:
await stream_audio(
    ENDPOINT_NAME,
    pcm,
    sample_rate,
    params={"keyterms_prompt": json.dumps(["LibriVox", "Bennet"]), "mode": "min_latency"},
    on_message=show,
)

The model package contains the speech model only, so options that need other AssemblyAI services, such as `redact_pii` or `speaker_labels`, are refused with an `Error` (close code 3006) rather than silently ignored:

In [ ]:
await stream_audio(ENDPOINT_NAME, pcm[: sample_rate * 2], sample_rate, params={"redact_pii": "true"}, on_message=show)

<a id='invoke'></a>
## 7. Whole-file transcription

`InvokeEndpoint` sends a complete file (WAV, FLAC, OGG or MP3) and returns every turn at once. Options go in `CustomAttributes` as a JSON object (up to 1024 characters). `InvokeEndpoint` allows 60 seconds per request, which suits files of about a minute; use asynchronous inference or batch transform for longer recordings.

In [ ]:
with open("data/input/sample-30s.wav", "rb") as f:
    audio_bytes = f.read()

response = smr.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="audio/wav",
    Body=audio_bytes,
    CustomAttributes=json.dumps({"keyterms_prompt": ["LibriVox"]}),
)
result = json.loads(response["Body"].read())

print(result["text"])
print()
print(f"{len(result['turns'])} turns, {result['audio_duration_ms']} ms of audio")

Each entry in `turns` is a final streaming `Turn`, with word-level timings in milliseconds:

In [ ]:
print(f"{'start':>7} {'end':>7}  word")
for word in result["turns"][0]["words"][:10]:
    print(f"{word['start']:>7} {word['end']:>7}  {word['text']}")

Invalid or unsupported options are answered with HTTP 400, raised by `boto3` as `ModelError`:

In [ ]:
try:
    smr.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="audio/wav",
        Body=audio_bytes,
        CustomAttributes=json.dumps({"speaker_labels": True}),
    )
except smr.exceptions.ModelError as e:
    print(e.response["OriginalStatusCode"], e.response["OriginalMessage"])

<a id='batch'></a>
## 8. Batch transform

Batch transform processes many files from Amazon S3 in one job. Each object is one record, and its result is written to the output prefix as `<filename>.out` in the response format above.

- Set `SplitType` to `None` so a file is never split.
- Batch hosts need `TransformAmiVersion="al2-ami-sagemaker-batch-gpu-535"`. Pass no `Environment`: SageMaker rejects it on a Marketplace package's job, and the container enables CUDA compatibility itself.
- Leave `MaxConcurrentTransforms` and `MaxPayloadInMB` unset to use the container's defaults.
- Batch transform cannot set `CustomAttributes`; to pass options, store each file as a JSON envelope `{"audio": "<base64>", "config": {...}}` and set `ContentType` to `application/json`.

Supported batch instance type: `ml.g5.2xlarge`.

In [ ]:
s3 = session.client("s3")
account = session.client("sts").get_caller_identity()["Account"]
BUCKET = f"sagemaker-{region}-{account}"  # the SageMaker default bucket name
with contextlib.suppress(s3.exceptions.BucketAlreadyOwnedByYou):
    if region == "us-east-1":
        s3.create_bucket(Bucket=BUCKET)
    else:
        s3.create_bucket(Bucket=BUCKET, CreateBucketConfiguration={"LocationConstraint": region})

PREFIX = "universal-3-6-pro-streaming"
for name in ("sample-10s.wav", "sample-30s.wav"):
    s3.upload_file(f"data/input/batch/{name}", BUCKET, f"{PREFIX}/batch-input/{name}")
S3_INPUT = f"s3://{BUCKET}/{PREFIX}/batch-input/"
S3_OUTPUT = f"s3://{BUCKET}/{PREFIX}/batch-output/"

TRANSFORM_JOB_NAME = f"universal-3-6-pro-streaming-{int(time.time())}"
sm.create_transform_job(
    TransformJobName=TRANSFORM_JOB_NAME,
    ModelName=MODEL_NAME,
    BatchStrategy="SingleRecord",
    ModelClientConfig={"InvocationsTimeoutInSeconds": 600, "InvocationsMaxRetries": 0},
    TransformInput={
        "DataSource": {"S3DataSource": {"S3DataType": "S3Prefix", "S3Uri": S3_INPUT}},
        "ContentType": "audio/wav",
        "CompressionType": "None",
        "SplitType": "None",
    },
    TransformOutput={"S3OutputPath": S3_OUTPUT, "Accept": "application/json", "AssembleWith": "None"},
    TransformResources={
        "InstanceType": "ml.g5.2xlarge",
        "InstanceCount": 1,
        "TransformAmiVersion": "al2-ami-sagemaker-batch-gpu-535",
    },
)
print(f"Started transform job: {TRANSFORM_JOB_NAME}")

In [ ]:
sm.get_waiter("transform_job_completed_or_stopped").wait(
    TransformJobName=TRANSFORM_JOB_NAME, WaiterConfig={"Delay": 30, "MaxAttempts": 120}
)
job = sm.describe_transform_job(TransformJobName=TRANSFORM_JOB_NAME)
print("Status:", job["TransformJobStatus"], job.get("FailureReason", ""))

for obj in s3.list_objects_v2(Bucket=BUCKET, Prefix=f"{PREFIX}/batch-output/").get("Contents", []):
    result = json.loads(s3.get_object(Bucket=BUCKET, Key=obj["Key"])["Body"].read())
    print(f"{obj['Key'].rsplit('/', 1)[-1]}: {result['text'][:100]}...")

<a id='metrics'></a>
## 9. Usage metrics

The software charge is metered by the amount of audio processed, in units of 8 seconds of audio (see the listing for the price per unit); batch transform is priced per instance-hour instead, with no per-audio charge. Every stream and request also writes its usage to the container's CloudWatch logs (the endpoint's or the transform job's) in Embedded Metric Format, so you can graph, alarm on and reconcile it in your own account: namespace `AssemblyAI/SageMakerStreaming`, metrics `ConsumedUnits` and `AudioDurationSeconds`, dimension `Operation` (`stream` or `file`).

Metrics appear in CloudWatch a few minutes after the requests that produced them.

In [ ]:
cloudwatch = session.client("cloudwatch")
end = datetime.now(timezone.utc)
for operation in ("stream", "file"):
    for metric in ("AudioDurationSeconds", "ConsumedUnits"):
        points = cloudwatch.get_metric_statistics(
            Namespace="AssemblyAI/SageMakerStreaming",
            MetricName=metric,
            Dimensions=[{"Name": "Operation", "Value": operation}],
            StartTime=end - timedelta(hours=3),
            EndTime=end,
            Period=10800,
            Statistics=["Sum"],
        )["Datapoints"]
        print(f"{operation:>6} {metric:<21} {sum(p['Sum'] for p in points):g}")

<a id='cleanup'></a>
## 10. Clean up

Endpoints bill for as long as they are running. Delete the endpoint when you are finished.

Deleting these resources does not cancel your AWS Marketplace subscription.

In [ ]:
sm.delete_endpoint(EndpointName=ENDPOINT_NAME)
sm.delete_endpoint_config(EndpointConfigName=ENDPOINT_CONFIG_NAME)
sm.delete_model(ModelName=MODEL_NAME)
print("Deleted endpoint, endpoint configuration, and model.")

<a id='unsubscribe'></a>
## 11. Unsubscribe (optional)

If you no longer need the model, cancel the subscription so you are not charged for future use:

1. Confirm no deployable model, live endpoint, or running batch transform job created from this model package remains in your account.
2. Open **Your Marketplace Software** on the AWS Marketplace website (or **AWS Marketplace > Manage subscriptions**), locate this product, and cancel the subscription.